# Lab 2-1：Adult Census Income 全连接神经网络

使用预处理后的 `X_train`、`Y_train`、`X_test`，将训练集按 8:2 分层划分，标准化后训练一个 `输入层 → 64 → 64 → 2` 的全连接神经网络。

训练 20 个 epoch，并按验证集最小 loss 保存最佳模型到 `MyModels/adult_Model.pth`；最终生成 `/kaggle/working/adult_output.csv`。

> 实际竞赛文件有 106 个输入特征，而实验 PDF 写的是 510 维。这里以实际文件为准，代码会自动读取输入维度。

### 阅读提示

Notebook 的主线是：**定位文件 → 清洗数据 → 划分并标准化 → 构建网络 → 训练并保存最佳模型 → 预测 → 生成提交文件**。建议按顺序运行，不要跳过中间单元格。

## 1. 导入依赖并定位数据

这一部分完成三件事：固定随机种子以便复现、选择 CPU/GPU、在 Kaggle 的 `/kaggle/input` 下自动寻找三个数据文件。`/kaggle/input` 只读，模型和提交文件写入 `/kaggle/working`。

In [1]:
# pathlib/os 用于处理 Kaggle 文件路径，random 用于固定随机性。
from pathlib import Path
import os
import random

# NumPy/Pandas 负责数据处理，PyTorch 负责神经网络训练。
import numpy as np
import pandas as pd
from IPython.display import display

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# 固定随机种子：重复运行时，数据划分和初始权重尽量保持一致。
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Kaggle 有可用 GPU 时使用 CUDA，否则自动使用 CPU。
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# INPUT_ROOT 是只读输入目录；WORKING_DIR 用于保存模型与提交文件。
INPUT_ROOT = Path(os.environ.get('LAB2_INPUT_ROOT', '/kaggle/input'))
WORKING_DIR = Path(os.environ.get('LAB2_WORKING_DIR', '/kaggle/working'))
WORKING_DIR.mkdir(parents=True, exist_ok=True)

# 三个逻辑文件名以及允许的紧凑写法。
required_keys = {'x_train', 'y_train', 'x_test'}
key_aliases = {'xtrain': 'x_train', 'ytrain': 'y_train', 'xtest': 'x_test'}

def dataset_key(path):
    # 同时接受 X_train、X_train.csv、x-train.txt 等 Kaggle 挂载名称。
    base_name = path.name.lower().split('.')[0]
    compact_name = ''.join(char for char in base_name if char.isalnum())
    return key_aliases.get(compact_name)

# 如果右侧没有 Add Input，/kaggle/input 中就不会出现竞赛数据。
if not INPUT_ROOT.exists():
    raise FileNotFoundError(f'输入根目录不存在：{INPUT_ROOT}。请先在右侧 Add Input 添加竞赛数据。')

# 递归扫描输入目录，因为 Kaggle 会把数据放进竞赛名称子目录。
all_input_files = sorted(
    [path for path in INPUT_ROOT.rglob('*') if path.is_file()],
    key=lambda path: str(path)
)
# 同时按目录和逻辑文件名记录匹配结果，防止误选其他数据集。
matches_by_dir = {}
matches_by_key = {key: [] for key in required_keys}
for path in all_input_files:
    key = dataset_key(path)
    if key is not None:
        matches_by_dir.setdefault(path.parent, {})[key] = path
        matches_by_key[key].append(path)

# 优先选择同一目录中同时包含三个文件的一组数据。
complete_groups = [
    paths for paths in matches_by_dir.values() if set(paths) == required_keys
]
if len(complete_groups) == 1:
    resolved_paths = complete_groups[0]
elif len(complete_groups) == 0 and all(len(matches_by_key[key]) == 1 for key in required_keys):
    # 兼容三个文件被分别放在不同子目录的情况。
    resolved_paths = {key: matches_by_key[key][0] for key in required_keys}
else:
    preview = [str(path.relative_to(INPUT_ROOT)) for path in all_input_files[:80]]
    raise FileNotFoundError(
        '无法唯一识别 X_train、Y_train、X_test。\n'
        f'输入根目录：{INPUT_ROOT}\n'
        f'识别结果：{matches_by_key}\n'
        f'实际文件（最多显示 80 个）：{preview}\n'
        '请确认右侧 Input 已添加本竞赛数据。'
    )

# 后续代码只使用这三个已经确认的真实路径。
X_TRAIN_PATH = resolved_paths['x_train']
Y_TRAIN_PATH = resolved_paths['y_train']
X_TEST_PATH = resolved_paths['x_test']

print('X_train：', X_TRAIN_PATH)
print('Y_train：', Y_TRAIN_PATH)
print('X_test ：', X_TEST_PATH)
print('运行设备：', DEVICE)

X_train： /kaggle/input/competitions/2026zjutest2/X_train
Y_train： /kaggle/input/competitions/2026zjutest2/Y_train
X_test ： /kaggle/input/competitions/2026zjutest2/X_test
运行设备： cpu


## 2. 读取并检查预处理数据

先用 Pandas 读取表格，再检查列是否对应、标签是否为 0/1。最后转换为神经网络更适合处理的 NumPy 数组：特征使用 `float32`，标签使用整数 `int64`。

In [2]:
# DataFrame 保留列名，便于检查训练集和测试集是否对齐。
X_train_df = pd.read_csv(X_TRAIN_PATH)
Y_train_df = pd.read_csv(Y_TRAIN_PATH)
X_test_df = pd.read_csv(X_TEST_PATH)

# 在训练之前尽早发现重复列、错位列或样本数量不一致。
if X_train_df.columns.duplicated().any() or X_test_df.columns.duplicated().any():
    raise ValueError('特征列名存在重复，无法安全对齐。')
if list(X_train_df.columns) != list(X_test_df.columns):
    raise ValueError('X_train 与 X_test 的列名或列顺序不一致。')
if Y_train_df.shape[1] != 1:
    raise ValueError(f'Y_train 应只有一列，实际为 {Y_train_df.shape[1]} 列。')
if len(X_train_df) != len(Y_train_df):
    raise ValueError('X_train 与 Y_train 的样本数不一致。')

# 无法转为数字的值先变成 NaN，再连同无穷值一起填为 0。
X_train_df = X_train_df.apply(pd.to_numeric, errors='coerce')
X_test_df = X_test_df.apply(pd.to_numeric, errors='coerce')
X_train_df = X_train_df.replace([np.inf, -np.inf], np.nan).fillna(0.0)
X_test_df = X_test_df.replace([np.inf, -np.inf], np.nan).fillna(0.0)

# CrossEntropyLoss 要求类别标签是整数，因此使用 int64。
y = pd.to_numeric(Y_train_df.iloc[:, 0], errors='raise').to_numpy(dtype=np.int64)
if not set(np.unique(y)).issubset({0, 1}):
    raise ValueError(f'标签必须为 0/1，实际出现：{np.unique(y)}')

# 神经网络通常使用 float32：精度足够，而且比 float64 更省内存。
X_all = X_train_df.to_numpy(dtype=np.float32, copy=True)
X_test_all = X_test_df.to_numpy(dtype=np.float32, copy=True)
# 不硬编码 PDF 中的维度，直接以实际数据列数作为输入层大小。
INPUT_SIZE = X_all.shape[1]

print('X_train:', X_all.shape)
print('Y_train:', y.shape)
print('X_test :', X_test_all.shape)
print('实际输入维度:', INPUT_SIZE)
print('标签分布:', dict(zip(*np.unique(y, return_counts=True))))
display(X_train_df.head())

X_train: (32561, 106)
Y_train: (32561,)
X_test : (16281, 106)
实际输入维度: 106
标签分布: {np.int64(0): np.int64(24720), np.int64(1): np.int64(7841)}


,age,fnlwgt,sex,capital_gain,capital_loss,hours_per_week,Federal-gov,Local-gov,Never-worked,Private,...,Puerto-Rico,Scotland,South,Taiwan,Thailand,Trinadad&Tobago,United-States,Vietnam,Yugoslavia,?_native_country
0,39,77516,1,2174,0,40,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
1,50,83311,1,0,0,13,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0
2,38,215646,1,0,0,40,0,0,0,1,...,0,0,0,0,0,0,1,0,0,0
3,53,234721,1,0,0,40,0,0,0,1,...,0,0,0,0,0,0,1,0,0,0
4,28,338409,0,0,0,40,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0


## 3. 8:2 分层划分与标准化

分层划分会让训练集和验证集保持近似的 0/1 类别比例。均值和标准差只从训练子集计算，避免把验证集信息泄漏到训练过程。

标准化公式为 `x' = (x - mean) / std`。它把不同量纲的特征拉到相近尺度，使梯度更新更加稳定。

In [3]:
# 分层划分：分别切分每个类别，再合并，避免验证集类别比例严重偏移。
def stratified_split_indices(labels, validation_ratio=0.2, seed=42):
    rng = np.random.default_rng(seed)
    train_parts, valid_parts = [], []
    for label in np.unique(labels):
        # 找出当前类别的全部样本下标，并随机打乱。
        indices = np.flatnonzero(labels == label).copy()
        rng.shuffle(indices)
        # 当前类别约 20% 进入验证集，其余进入训练集。
        valid_count = max(1, int(round(len(indices) * validation_ratio)))
        valid_parts.append(indices[:valid_count])
        train_parts.append(indices[valid_count:])
    train_indices = np.concatenate(train_parts)
    valid_indices = np.concatenate(valid_parts)
    rng.shuffle(train_indices)
    rng.shuffle(valid_indices)
    return train_indices, valid_indices

train_idx, valid_idx = stratified_split_indices(y, validation_ratio=0.2, seed=SEED)

# 只用训练子集计算每一列的均值和标准差，避免验证集信息泄漏。
feature_mean = X_all[train_idx].mean(axis=0, dtype=np.float64).astype(np.float32)
feature_std = X_all[train_idx].std(axis=0, dtype=np.float64).astype(np.float32)
# 常数列的标准差为 0；改为 1 后，该列标准化结果自然为 0。
feature_std[feature_std < 1e-8] = 1.0

# 训练、验证和测试集必须复用同一组训练统计量。
X_train = ((X_all[train_idx] - feature_mean) / feature_std).astype(np.float32)
X_valid = ((X_all[valid_idx] - feature_mean) / feature_std).astype(np.float32)
X_test = ((X_test_all - feature_mean) / feature_std).astype(np.float32)
y_train = y[train_idx]
y_valid = y[valid_idx]

print('训练子集:', X_train.shape, y_train.shape)
print('验证子集:', X_valid.shape, y_valid.shape)
print('测试集  :', X_test.shape)

训练子集: (26049, 106) (26049,)
验证子集: (6512, 106) (6512,)
测试集  : (16281, 106)


## 4. 构建 PDF 指定的全连接神经网络

`DataLoader` 每次取 256 个样本送入网络。两个隐藏层各有 64 个神经元，并使用 ReLU 加入非线性；最后输出两个 logits，分别对应收入 `<=50K` 和 `>50K`。因为 `CrossEntropyLoss` 内部已经处理 Softmax，所以输出层不再添加激活函数。

In [4]:
# 每次用 256 个样本估计梯度，兼顾速度和显存/内存占用。
BATCH_SIZE = 256
train_generator = torch.Generator().manual_seed(SEED)

# TensorDataset 把特征和标签绑定；测试集没有标签，所以只有特征。
train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
valid_dataset = TensorDataset(torch.from_numpy(X_valid), torch.from_numpy(y_valid))
test_dataset = TensorDataset(torch.from_numpy(X_test))

# 训练集需要 shuffle；验证集和测试集保持固定顺序。
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=0, generator=train_generator
)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# 数据流：106维输入（实际读取）→ 64 → 64 → 2个类别分数。
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(INPUT_SIZE, 64),
    nn.ReLU(),  # 加入非线性，否则多层 Linear线性层仍等价于一层。
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 2),  # 输出两个 logits，不在这里手动做 Softmax。
).to(DEVICE)

# 交叉熵衡量分类错误；Adam 根据梯度自适应更新权重。
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
print(model)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=106, out_features=64, bias=True)
  (2): ReLU()
  (3): Linear(in_features=64, out_features=64, bias=True)
  (4): ReLU()
  (5): Linear(in_features=64, out_features=2, bias=True)
)


## 5. 训练 20 个 epoch，并保存验证 loss 最小的模型

一个 epoch 表示模型完整看过一次训练集。训练阶段执行反向传播并更新参数；验证阶段只计算 loss 和 accuracy。验证 loss 创下新低时保存模型，避免最后一轮已经过拟合。

In [5]:
# 用同一个函数完成训练或验证；training=True 时才计算梯度并更新参数。
def run_epoch(data_loader, training):
    model.train(training)  # 在 train/eval 模式之间切换。
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    # DataLoader 每次提供一个小批量的特征和真实标签。
    for features, labels in data_loader:
        features = features.to(DEVICE)
        labels = labels.to(DEVICE)

        if training:
            # PyTorch 默认累加梯度，因此每个批次开始前必须清零。
            optimizer.zero_grad(set_to_none=True)

        # 验证阶段关闭梯度记录，可节省内存和计算量。
        with torch.set_grad_enabled(training):
            logits = model(features)  # 前向传播，形状为 [batch_size, 2]。
            loss = criterion(logits, labels)
            if training:
                # backward 计算梯度，step 根据梯度更新所有权重和偏置。
                loss.backward()
                optimizer.step()

        # 累计每个样本的 loss 和正确数量，最后得到整轮平均值。
        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_count += batch_size

    return total_loss / total_count, total_correct / total_count

# 模型最多完整学习训练集 20 次。
EPOCHS = 20
MODEL_DIR = WORKING_DIR / 'MyModels'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / 'adult_Model.pth'

# best_valid_loss 从无穷大开始，第一轮一定会成为暂时最佳模型。
best_valid_loss = float('inf')
best_epoch = 0
history = []

for epoch in range(1, EPOCHS + 1):
    # 先学习训练集，再用不参与更新的验证集检查泛化效果。
    train_loss, train_acc = run_epoch(train_loader, training=True)
    valid_loss, valid_acc = run_epoch(valid_loader, training=False)

    # 验证 loss 创新低才覆盖模型文件，避免保存已经过拟合的最后一轮。
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_epoch = epoch
        # checkpoint 同时保存权重和本次训练所需的说明数据。
        torch.save({
            'model_state_dict': model.state_dict(),
            'input_size': INPUT_SIZE,
            'best_epoch': best_epoch,
            'best_valid_loss': best_valid_loss,
            'feature_mean': feature_mean,
            'feature_std': feature_std,
        }, MODEL_PATH)

    # 保存每轮指标，便于最后查看或绘制学习曲线。
    history.append({
        'epoch': epoch, 'train_loss': train_loss, 'train_acc': train_acc,
        'valid_loss': valid_loss, 'valid_acc': valid_acc
    })
    print(
        f'Epoch {epoch:02d}/{EPOCHS} | '
        f'train loss {train_loss:.4f}, acc {train_acc:.4f} | '
        f'valid loss {valid_loss:.4f}, acc {valid_acc:.4f}'
    )

history_df = pd.DataFrame(history)
print(f'最佳模型：epoch {best_epoch}，验证 loss {best_valid_loss:.4f}')
print('模型保存到：', MODEL_PATH)
display(history_df.tail())

Epoch 01/20 | train loss 0.4168, acc 0.8009 | valid loss 0.3358, acc 0.8432
Epoch 02/20 | train loss 0.3282, acc 0.8463 | valid loss 0.3206, acc 0.8504
Epoch 03/20 | train loss 0.3149, acc 0.8541 | valid loss 0.3168, acc 0.8490
Epoch 04/20 | train loss 0.3071, acc 0.8572 | valid loss 0.3168, acc 0.8533
Epoch 05/20 | train loss 0.3019, acc 0.8604 | valid loss 0.3138, acc 0.8546
Epoch 06/20 | train loss 0.2978, acc 0.8612 | valid loss 0.3157, acc 0.8557
Epoch 07/20 | train loss 0.2945, acc 0.8624 | valid loss 0.3168, acc 0.8514
Epoch 08/20 | train loss 0.2914, acc 0.8650 | valid loss 0.3200, acc 0.8546
Epoch 09/20 | train loss 0.2894, acc 0.8656 | valid loss 0.3174, acc 0.8524
Epoch 10/20 | train loss 0.2863, acc 0.8671 | valid loss 0.3201, acc 0.8512
Epoch 11/20 | train loss 0.2837, acc 0.8689 | valid loss 0.3252, acc 0.8509
Epoch 12/20 | train loss 0.2831, acc 0.8682 | valid loss 0.3186, acc 0.8529
Epoch 13/20 | train loss 0.2804, acc 0.8705 | valid loss 0.3256, acc 0.8507
Epoch 14/20 

,epoch,train_loss,train_acc,valid_loss,valid_acc
15,16,0.274333,0.872855,0.332822,0.849201
16,17,0.273280,0.874045,0.331774,0.852273
17,18,0.271676,0.874928,0.335428,0.844902
18,19,0.271059,0.874506,0.336077,0.847973
19,20,0.267240,0.875965,0.337610,0.848280


## 6. 载入最佳模型并预测测试集

这里加载的不是第 20 轮模型，而是训练过程中验证 loss 最小的模型。每个测试样本会得到两个 logits，`argmax(dim=1)` 取分数更高的类别作为最终的 0/1 预测。

In [6]:
# 这个 checkpoint 是上一个单元格亲自生成的可信文件。
# weights_only=False 用于兼容其中保存的 NumPy 均值/标准差；不要对未知来源模型这样做。
checkpoint = torch.load(
    MODEL_PATH,
    map_location=DEVICE,
    weights_only=False
)
# 把最佳轮次的参数装回相同结构的网络，并切换到预测模式。
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

prediction_parts = []
# 预测不需要反向传播，因此关闭梯度记录。
with torch.no_grad():
    for (features,) in test_loader:
        logits = model(features.to(DEVICE))
        # dim=1 表示在每个样本的两个类别分数中取最大值位置。
        prediction_parts.append(logits.argmax(dim=1).cpu().numpy())

# 将各批次预测按原始测试顺序重新拼成一个数组。
test_predictions = np.concatenate(prediction_parts).astype(np.int64)
print('预测数量：', len(test_predictions))
print('预测分布：', dict(zip(*np.unique(test_predictions, return_counts=True))))

预测数量： 16281
预测分布： {np.int64(0): np.int64(13034), np.int64(1): np.int64(3247)}


## 7. 生成 Kaggle 提交文件

实验 PDF 的提交示例从 `id=1` 开始，因此这里严格生成 `1, 2, ..., 16281`，而不是从 0 开始。断言用于在保存前自动检查列名、行数、ID 和标签范围，避免把格式错误的文件提交到 Kaggle。

In [7]:
# Kaggle 要求两列：从 1 开始的 id，以及对应的 0/1 预测标签。
submission = pd.DataFrame({
    'id': np.arange(1, len(test_predictions) + 1, dtype=np.int64),
    'label': test_predictions,
})

# 以下断言相当于提交前的自动质检；任何条件不满足都会立即报错。
expected_ids = np.arange(1, len(X_test_df) + 1, dtype=np.int64)
assert list(submission.columns) == ['id', 'label']
assert len(submission) == len(X_test_df) == 16281
assert np.array_equal(submission['id'].to_numpy(), expected_ids)
assert submission['id'].is_unique
assert set(submission['label'].unique()).issubset({0, 1})
assert not submission.isna().any().any()

# index=False 防止 Pandas 额外写入一列 0,1,2... 的行索引。
OUTPUT_PATH = WORKING_DIR / 'adult_output.csv'
submission.to_csv(OUTPUT_PATH, index=False)

print('提交文件已生成：', OUTPUT_PATH)
print('行数（不含表头）：', len(submission))
display(submission.head(10))
display(submission.tail(3))

提交文件已生成： /kaggle/working/adult_output.csv
行数（不含表头）： 16281


,id,label
0,1,0
1,2,0
2,3,1
3,4,1
4,5,0
5,6,0
6,7,0
7,8,1
8,9,0
9,10,0


,id,label
16278,16279,1
16279,16280,0
16280,16281,1
